# Repo 1 : Data & Features

**Objectif** : Préparer les données brutes pour l'analyse et la modélisation.  
- Chargement du dataset bancaire.
- Exploration rapide.
- Nettoyage & standardisation.
- Feature engineering.
- Export des datasets segmentés (profil, campagne, finance, macro).

**Données** : `bank-additional-full.csv` (fichier CSV fourni).



2. Imports & setup


In [ ]:
import os
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Créer dossier outputs
PROJECT_ROOT = Path(os.getcwd())
OUT_DIR = PROJECT_ROOT / 'outputs'
OUT_DIR.mkdir(exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Outputs folder:', OUT_DIR)


3. Chargement des données

In [ ]:
data_path = PROJECT_ROOT / "data/bank-additional-full.csv"
df = pd.read_csv(data_path, sep=None, engine='python')
df['id'] = df.index + 1

# Aperçu
display(df.head())
print(df.info())
print(df.describe())


4. Résumé des colonnes

In [ ]:
# Générer un tableau de synthèse des colonnes
cols_summary = [
    {'col': 'age', 'dtype': 'int', 'utilité': 'Distribution des âges', 'importance': 'Forte', 'remarque': 'Vérifier présence d’âges extrêmes'},
    {'col': 'job', 'dtype': 'cat', 'utilité': 'Répartition par profession', 'importance': 'Moyenne à forte', 'remarque': 'Encodage nécessaire'},
    {'col': 'marital', 'dtype': 'cat', 'utilité': 'Statut marital', 'importance': 'Moyenne', 'remarque': 'Regrouper éventuellement en couples / célibataires'},
    # ... toutes les autres colonnes
    {'col': 'y', 'dtype': 'cat', 'utilité': 'Variable cible', 'importance': 'Très forte', 'remarque': 'À prédire, encoder “yes”=1 / “no”=0'}
]

cols_df = pd.DataFrame(cols_summary)
cols_df.to_csv(OUT_DIR / 'columns_summary.csv', index=False)
display(cols_df)


5. Prétraitement

In [ ]:
def standardize_missing(df):
    df = df.copy()
    for c in df.select_dtypes(include=['object']).columns:
        df[c] = df[c].replace(['unknown','nonexistent'], np.nan)
    return df

df_clean = standardize_missing(df)

# Vérifier les missing values
display(df_clean.isna().sum().sort_values(ascending=False).head(10))


6. Découpage par groupes de colonnes

In [ ]:
profil_cols = ['id','age','job','marital','education','default','housing','loan']
campagne_cols = ['id','contact','month','day_of_week','duration','campaign','pdays','previous','poutcome','y']
macro_cols = ['id','emp.var.rate','cons.price.idx','cons.conf.idx','euribor3m','nr.employed']

df_profil = df_clean[profil_cols].copy()
df_campagne = df_clean[campagne_cols].copy()
df_macro = df_clean[macro_cols].copy()

# Export CSVs
df_profil.to_csv(OUT_DIR / 'profil_cols.csv', index=False)
df_campagne.to_csv(OUT_DIR / 'campagne_cols.csv', index=False)
df_macro.to_csv(OUT_DIR / 'macro_cols.csv', index=False)
print("CSV files exported to", OUT_DIR)

7. Feature Engineering

In [ ]:
df_fe = df_clean.copy()

# Binning âge
age_bins = [0, 25, 35, 50, 65, 120]
age_labels = ['<25','25-34','35-49','50-64','65+']
df_fe['age_bin'] = pd.cut(df_fe['age'], bins=age_bins, labels=age_labels, include_lowest=True)

# pdays
df_fe['pdays_never_contacted'] = (df_fe['pdays'] == 999).astype(int)
df_fe['pdays_since'] = df_fe['pdays'].replace(999, np.nan)

# log duration
df_fe['duration_log1p'] = np.log1p(df_fe['duration'])

# Mois → trimestre → saison
month_map = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}
df_fe['month_num'] = df_fe['month'].map(month_map)
df_fe['quarter'] = pd.to_datetime(df_fe['month_num'], format='%m', errors='coerce').dt.quarter
df_fe['season'] = df_fe['quarter'].map({1:'winter', 2:'spring', 3:'summer', 4:'autumn'})

# Interaction simple : jeune + appel court
df_fe['young_short_call'] = ((df_fe['age'] < 35) & (df_fe['duration'] < df_fe['duration'].median())).astype(int)

# Encodage ordinal éducation
edu_order = ['illiterate','basic.4y','basic.6y','basic.9y','high.school','university.degree','professional.course']
df_fe['education_ord'] = df_fe['education'].astype(pd.CategoricalDtype(categories=edu_order, ordered=True)).cat.codes.replace(-1,np.nan)

# Variable cible binaire
df_fe['y_bin'] = df_fe['y'].map({'yes':1,'no':0})

# Export final
df_fe.to_csv(OUT_DIR / 'up_data.csv', index=False)
display(df_fe.head())
